Importujemy biblioteki do danych, modelu, wykresów i metryk, przy czym scikit-image musi być zainstalowane w środowisku kernela.

In [ ]:
from datetime import datetime
from importlib.metadata import version
from pathlib import Path
import json
import platform
import random
from time import perf_counter

import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.ticker import MaxNLocator
from IPython.display import display
import numpy as np
import pandas as pd
import torch
from datasets import Dataset as HFDataset, load_from_disk
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, SequentialSampler
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm


Odnajdujemy katalog projektu na podstawie wspólnego modułu wczytywania danych.


In [ ]:
project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks/matg/dataloading_demo.py").is_file()
    ),
    None,
)


Wybieramy tryb train, resume albo evaluate, przy czym RUN_PATH wskazujemy dla istniejącego uruchomienia, a EPOCHS oznacza łączną docelową liczbę epok.


In [ ]:
RUN_MODE = "train"
RUN_PATH = None

MODEL_NAME = "espcn_wide_v1"
MODEL_PARAMS = {"hidden_channels": 128}
SCALE_FACTOR = 2
HR_SIZE = 256

SEED = 42
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 64
TRAIN_LIMIT = 2560
VAL_LIMIT = 1280
AUGMENT_TRAIN = True
NUM_WORKERS = 0

EPOCHS = 150
LEARNING_RATE = 0.005
LR_STEP_SIZE = 10
LR_GAMMA = 0.75

RUN_TEST = False
SHOW_PROGRESS = True
PREVIEW_INDICES = [5, 6, 7, 8, 9]
CASE_COUNT = 10
INFERENCE_TILE_SIZE = 256
INFERENCE_OVERLAP = None


if project_root is None:
    raise FileNotFoundError("Uruchom notebook z katalogu projektu lub podkatalogu.")

DATASET_PATH = project_root / "notebooks/igoh/data/wikiart_train_processed"
EXPERIMENTS_PATH = project_root / "notebooks/igoh/models_weights/experiments"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Używamy oszczędnego koloru i bezpośrednich etykiet według zasad [Stephen Few](https://www.perceptualedge.com/articles/visual_business_intelligence/rules_for_using_color.pdf) oraz [Cole Nussbaumer Knaflic](https://www.storytellingwithdata.com/blog/two-tips-for-better-graphs), wyróżniając model niebieskim i pozostawiając kontekst w szarości.

In [ ]:
BLUE = "#1F5A85"
GRAY = "#8B9299"
TEXT = "#374151"
GRID = "#E5E7EB"

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "text.color": TEXT,
        "axes.labelcolor": TEXT,
        "xtick.color": TEXT,
        "ytick.color": TEXT,
        "font.size": 10,
        "axes.titlelocation": "left",
        "axes.titlesize": 12,
        "figure.dpi": 110,
    }
)


def style_axis(axis: Axes, grid_axis: str = "y") -> None:
    """Usuń zbędne obramowania i pozostaw delikatną siatkę wspierającą odczyt."""

    for side in ("top", "right", "left"):
        axis.spines[side].set_visible(False)

    axis.spines["bottom"].set_color(GRID)
    axis.tick_params(length=0, pad=8)
    axis.grid(axis=grid_axis, color=GRID, linewidth=0.7)
    axis.set_axisbelow(True)


Wczytujemy przygotowany WikiArt i odtwarzamy podział 80/10/10 z matg bez zmiany kolejności rekordów przed podziałem.

In [ ]:
def load_wikiart(path: Path) -> HFDataset:
    """Wczytaj lokalną bazę po preprocessingu bez ponownego pobierania."""

    if not (path / "dataset_info.json").is_file():
        raise FileNotFoundError(f"Brak przygotowanego zbioru: {path}")

    dataset = load_from_disk(str(path))

    if not isinstance(dataset, HFDataset) or "image" not in dataset.column_names:
        raise ValueError("Oczekiwano pojedynczego Dataset z kolumną image.")

    return dataset


def split_wikiart(
    dataset: HFDataset,
    seed: int,
) -> tuple[HFDataset, HFDataset, HFDataset]:
    """Odtwórz podział matg; zgodność wymaga identycznej bazy i kolejności."""

    split = dataset.train_test_split(test_size=0.2, seed=seed)
    holdout = split["test"].train_test_split(test_size=0.5, seed=seed)

    return split["train"], holdout["train"], holdout["test"]


Tylko w treningu losowo odbijamy HR poziomo przed utworzeniem LR, dzięki czemu wejście i wzorzec pozostają zgodne, a walidacja i test są niezmienne.

In [ ]:
class SuperResolutionDataset(Dataset[tuple[Tensor, Tensor]]):
    """Twórz pomniejszone wejście LR i referencję HR z tego samego obrazu."""

    def __init__(
        self,
        images: HFDataset,
        hr_size: int,
        scale_factor: int,
        augment: bool = False,
    ) -> None:
        """Zachowaj źródło i rozmiary par bez kopiowania obrazów."""

        if scale_factor <= 1 or hr_size <= 0 or hr_size % scale_factor != 0:
            raise ValueError("Niepoprawny rozmiar HR lub współczynnik skali.")

        self.images = images.select_columns(["image"])
        self.hr_size = hr_size
        self.lr_size = hr_size // scale_factor
        self.augment = augment

    def __len__(self) -> int:
        """Zwróć liczbę dostępnych par."""

        return len(self.images)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor]:
        """Zdekoduj obraz i pomniejsz go bicubic, zachowując oryginał jako HR."""

        hr_image = self.images[index]["image"].convert("RGB")

        if hr_image.size != (self.hr_size, self.hr_size):
            raise ValueError(
                f"Obraz {index}: rozmiar {hr_image.size}, "
                f"oczekiwano {self.hr_size}×{self.hr_size}."
            )

        if self.augment and torch.rand(()).item() < 0.5:
            hr_image = TF.hflip(hr_image)

        lr_image = hr_image.resize(
            (self.lr_size, self.lr_size),
            resample=Image.Resampling.BICUBIC,
        )

        return TF.to_tensor(lr_image), TF.to_tensor(hr_image)


Definiujemy DataLoader z powtarzalną kolejnością losowania i domyślnie bez procesów roboczych, aby działał w notebooku na Windows.

In [ ]:
def make_loader(
    dataset: Dataset[tuple[Tensor, Tensor]],
    batch_size: int,
    shuffle: bool,
    seed: int,
    num_workers: int = 0,
) -> DataLoader:
    """Grupuj pary w partie, losując kolejność tylko wtedy, gdy jest to wymagane."""

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        generator=torch.Generator().manual_seed(seed),
        pin_memory=torch.cuda.is_available(),
    )


Zachowujemy dwuwarstwowy wariant espcn_wide_v1 z kontekstem LR równym 3 i nadajemy nową nazwę wersji przy zmianach budowy niezgodnych z jego wagami.


In [ ]:
class ESPCNWide(nn.Module):
    """Odtwarzaj RGB za pomocą szerokiej warstwy cech i PixelShuffle."""

    def __init__(self, scale_factor: int = 2, hidden_channels: int = 128) -> None:
        """Zbuduj dwie konwolucje i określ kontekst potrzebny przy podziale obrazu."""

        super().__init__()

        if scale_factor not in (2, 4) or hidden_channels <= 0:
            raise ValueError("Wybierz skalę 2 lub 4 i dodatnią liczbę kanałów.")

        self.scale_factor = scale_factor
        self.required_overlap = 3

        self.layers = nn.Sequential(
            nn.Conv2d(3, hidden_channels, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv2d(hidden_channels, 3 * scale_factor**2, kernel_size=3, padding=1),
            nn.PixelShuffle(scale_factor),
        )

    def forward(self, lr: Tensor) -> Tensor:
        """Zamień partię LR (N, 3, H, W) w SR (N, 3, skala·H, skala·W)."""

        return self.layers(lr)


Tworzymy model na podstawie zapisanej nazwy i parametrów, dodając przyszłe architektury jako osobne klasy i kolejne przypadki wyboru.

In [ ]:
def build_model(name: str, params: dict[str, int]) -> nn.Module:
    """Odtwórz wskazany wariant bez zgadywania architektury na podstawie wag."""

    if name == "espcn_wide_v1":
        return ESPCNWide(**params)

    raise ValueError(f"Nieznana architektura: {name}.")


Zapisujemy środowisko obliczeniowe i ustawiamy losowanie, aby wznowienie w tym samym środowisku zachowało kolejność danych i augmentacji.

In [ ]:
def environment_info(device: torch.device) -> dict[str, object]:
    """Opisz biblioteki i urządzenie istotne dla odtwarzania treningu."""

    return {
        "python": platform.python_version(),
        "torch": str(torch.__version__),
        "numpy": np.__version__,
        "pillow": version("pillow"),
        "datasets": version("datasets"),
        "torchvision": version("torchvision"),
        "device_type": device.type,
        "device_name": (
            torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu"
        ),
        "cuda_devices": torch.cuda.device_count() if device.type == "cuda" else 0,
        "cpu_threads": torch.get_num_threads(),
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
    }


def seed_everything(seed: int, device: torch.device) -> None:
    """Ustaw początek losowania i powtarzalny wybór algorytmów cuDNN."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)


Tworzymy osobny katalog każdego treningu i zapisujemy pliki przez podmianę, aby przerwany zapis nie zniszczył poprzedniej kompletnej wersji.

In [ ]:
def save_csv(frame: pd.DataFrame, path: Path) -> None:
    """Zastąp CSV dopiero po zapisaniu całej tabeli do pliku tymczasowego."""

    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def create_run(base_path: Path, config: dict[str, object]) -> Path:
    """Utwórz nowy katalog z konfiguracją bez nadpisywania istniejącego treningu."""

    architecture = config["architecture"]
    scale = architecture["params"]["scale_factor"]
    stamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S_%f")
    name = f"{architecture['name']}_x{scale}_{config['loss_name'].lower()}_{stamp}"

    base_path.mkdir(parents=True, exist_ok=True)
    run_path = base_path / name
    run_path.mkdir(exist_ok=False)

    temporary = run_path / "config.json.tmp"
    temporary.write_text(
        json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    temporary.replace(run_path / "config.json")

    return run_path


def load_run_config(run_path: Path) -> dict[str, object]:
    """Wczytaj ustawienia uruchomienia w nowym formacie checkpointów."""

    config = json.loads((run_path / "config.json").read_text(encoding="utf-8"))

    if config.get("format_version") != 1:
        raise ValueError(
            "Nieobsługiwana wersja konfiguracji; stare pliki nie są migrowane."
        )

    return config


Zachowujemy stan losowania i generatorów obu loaderów bez obiektów NumPy wymagających wczytywania checkpointu w trybie weights_only=False.

In [ ]:
def capture_rng_state(
    loaders: tuple[DataLoader, DataLoader], device: torch.device
) -> dict[str, object]:
    """Zachowaj losowanie po ukończonej epoce przy NUM_WORKERS=0."""

    numpy_state = np.random.get_state()

    return {
        "python": random.getstate(),
        "numpy": [numpy_state[0], numpy_state[1].tolist(), *numpy_state[2:]],
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],
        "loaders": [loader.generator.get_state() for loader in loaders],
    }


def restore_rng_state(
    state: dict[str, object],
    loaders: tuple[DataLoader, DataLoader],
    device: torch.device,
) -> None:
    """Przywróć losowanie dopiero po utworzeniu modelu i loaderów."""

    numpy_state = state["numpy"]
    random.setstate(state["python"])
    np.random.set_state(
        (numpy_state[0], np.asarray(numpy_state[1], dtype=np.uint32), *numpy_state[2:])
    )
    torch.set_rng_state(state["torch"])

    if device.type == "cuda":
        torch.cuda.set_rng_state_all(state["cuda"])

    for loader, generator_state in zip(loaders, state["loaders"], strict=True):
        loader.generator.set_state(generator_state)


Trenujemy jedną epokę z MSE na surowym wyjściu sieci, bez ograniczania wartości do zakresu obrazu.


In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    show_progress: bool = True,
) -> float:
    """Przejdź raz przez trening i zwróć średnią stratę na przetworzony obraz."""

    model.train()
    total_loss = torch.zeros((), device=device)
    image_count = 0

    for lr, hr in tqdm(
        loader,
        desc="Trening",
        unit="partia",
        leave=False,
        mininterval=0.5,
        disable=not show_progress,
    ):
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        sr = model(lr)
        loss = criterion(sr, hr)

        loss.backward()
        optimizer.step()

        batch_size = lr.shape[0]
        total_loss += loss.detach() * batch_size
        image_count += batch_size

    if image_count == 0:
        raise ValueError("Brak obrazów do treningu.")

    return (total_loss / image_count).item()


Walidujemy na surowym wyjściu sieci tą samą stratą co trening, podczas gdy końcowe metryki obrazu liczymy osobno po ograniczeniu do [0, 1].


In [ ]:
@torch.inference_mode()
def validate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    show_progress: bool = True,
) -> float:
    """Oceń bieżące wagi na danych walidacyjnych i zwróć średnią stratę na obraz."""

    model.eval()
    total_loss = torch.zeros((), device=device)
    image_count = 0

    for lr, hr in tqdm(
        loader,
        desc="Walidacja",
        unit="partia",
        leave=False,
        mininterval=0.5,
        disable=not show_progress,
    ):
        lr = lr.to(device, non_blocking=True)
        hr = hr.to(device, non_blocking=True)

        sr = model(lr)
        loss = criterion(sr, hr)

        batch_size = lr.shape[0]
        total_loss += loss * batch_size
        image_count += batch_size

    if image_count == 0:
        raise ValueError("Brak obrazów do walidacji.")

    return (total_loss / image_count).item()


Zapisujemy checkpoint formatu 1 przez plik tymczasowy, zachowując wagi, opis architektury i stan potrzebny do kontynuowania treningu.


In [ ]:
def save_checkpoint(path: Path, checkpoint: dict[str, object]) -> None:
    """Podmień checkpoint dopiero po ukończeniu zapisu nowej wersji."""

    temporary = path.with_name(path.name + ".tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(path)


def read_checkpoint(path: Path) -> dict[str, object]:
    """Odczytaj nowy format i odrzuć nieobsługiwane starsze checkpointy."""

    checkpoint = torch.load(path, map_location="cpu", weights_only=True)

    if checkpoint.get("format_version") != 1:
        raise ValueError("Nieobsługiwany checkpoint; wymagany format_version=1.")
    if checkpoint["architecture"] != checkpoint["config"]["architecture"]:
        raise ValueError("Opis architektury nie zgadza się z konfiguracją.")

    epoch = checkpoint["epoch"]
    history = checkpoint["history"]

    if epoch < 1 or not 1 <= checkpoint["best_epoch"] <= epoch:
        raise ValueError("Niepoprawna numeracja epok w checkpointcie.")
    if any(
        len(history[key]) != epoch
        for key in ("train_loss", "val_loss", "learning_rate", "epoch_seconds")
    ):
        raise ValueError("Historia nie odpowiada liczbie zapisanych epok.")

    return checkpoint


Dopisujemy każdą ukończoną epokę do historii i zapamiętujemy najlepszy wynik według walidacyjnego MSE.


In [ ]:
def initial_training_state() -> dict[str, object]:
    """Zacznij nową historię i wybór najlepszej epoki."""

    return {
        "epoch": 0,
        "best_epoch": 0,
        "best_val_loss": float("inf"),
        "history": {
            "train_loss": [],
            "val_loss": [],
            "learning_rate": [],
            "epoch_seconds": [],
        },
    }


def record_epoch(
    state: dict[str, object],
    train_loss: float,
    val_loss: float,
    learning_rate: float,
    epoch_seconds: float,
) -> None:
    """Dopisz straty, użyty LR i czas treningu z walidacją bez zapisu plików."""

    if not np.isfinite(train_loss) or not np.isfinite(val_loss):
        raise ValueError("Strata nie jest skończona; sprawdź dane i trening.")

    for key, value in (
        ("train_loss", train_loss),
        ("val_loss", val_loss),
        ("learning_rate", learning_rate),
        ("epoch_seconds", epoch_seconds),
    ):
        state["history"][key].append(value)

    state["epoch"] += 1

    if val_loss < state["best_val_loss"]:
        state["best_val_loss"] = val_loss
        state["best_epoch"] = state["epoch"]


Po każdej epoce zapisujemy najpierw last.pt, następnie ewentualny best.pt i pełną historię CSV, niezależnie od poprawy walidacji.

In [ ]:
def save_history(history: dict[str, list[float]], run_path: Path) -> None:
    """Wyeksportuj wszystkie ukończone epoki do jednej czytelnej tabeli."""

    frame = pd.DataFrame(history)
    frame.insert(0, "epoch", np.arange(1, len(frame) + 1))
    save_csv(frame, run_path / "history.csv")


def save_epoch(
    run_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.StepLR,
    config: dict[str, object],
    state: dict[str, object],
    loaders: tuple[DataLoader, DataLoader],
    device: torch.device,
) -> None:
    """Utrwal pełny stan ukończonej epoki i oddziel najlepszy model od ostatniego."""

    checkpoint = {
        "format_version": 1,
        "architecture": config["architecture"],
        "config": config,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "val_loss": state["history"]["val_loss"][-1],
        "rng_state": capture_rng_state(loaders, device),
        **state,
    }

    save_checkpoint(run_path / "last.pt", checkpoint)

    if state["best_epoch"] == state["epoch"]:
        save_checkpoint(run_path / "best.pt", checkpoint)

    save_history(state["history"], run_path)


Przy ponownym otwarciu traktujemy last.pt jako źródło pełnej historii i naprawiamy pliki, których zapis mógł zostać przerwany.

In [ ]:
def recover_run_artifacts(run_path: Path, checkpoint: dict[str, object]) -> None:
    """Odtwórz CSV i najlepszy zapis, jeżeli przerwano zapis ostatniej epoki."""

    if checkpoint["best_epoch"] == checkpoint["epoch"]:
        save_checkpoint(run_path / "best.pt", checkpoint)
    else:
        best = read_checkpoint(run_path / "best.pt")

        if (
            best["epoch"] != checkpoint["best_epoch"]
            or best["val_loss"] != checkpoint["best_val_loss"]
            or best["config"] != checkpoint["config"]
        ):
            raise ValueError("best.pt nie odpowiada stanowi uruchomienia w last.pt.")

    save_history(checkpoint["history"], run_path)


Wznawiamy z last.pt w tym samym środowisku i przy NUM_WORKERS=0, odtwarzając optimizer, scheduler oraz losowanie po inicjalizacji wszystkich obiektów.

In [ ]:
def restore_training(
    run_path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.StepLR,
    config: dict[str, object],
    loaders: tuple[DataLoader, DataLoader],
    device: torch.device,
) -> dict[str, object]:
    """Odtwórz koniec epoki bez resetowania harmonogramu i najlepszego wyniku."""

    checkpoint = read_checkpoint(run_path / "last.pt")

    if checkpoint["config"] != config:
        raise ValueError(
            "Konfiguracja różni się od checkpointu; utwórz nowy eksperyment."
        )
    if config["environment"] != environment_info(device):
        raise ValueError("Wznowienie wymaga tego samego środowiska i typu urządzenia.")
    if any(loader.num_workers != 0 for loader in loaders):
        raise ValueError("Odtwarzanie losowania obsługujemy przy NUM_WORKERS=0.")

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    recover_run_artifacts(run_path, checkpoint)
    restore_rng_state(checkpoint["rng_state"], loaders, device)

    return {
        key: checkpoint[key]
        for key in ("epoch", "best_epoch", "best_val_loss", "history")
    }


Trenujemy do docelowego numeru epoki lub wznawiamy ostatni zapis, nie pozwalając przypadkowo rozpocząć nowego treningu w zajętym katalogu.


In [ ]:
def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.StepLR,
    device: torch.device,
    epochs: int,
    run_path: Path,
    config: dict[str, object],
    resume: bool = False,
    show_progress: bool = True,
) -> dict[str, list[float]]:
    """Ucz do łącznego celu epok, zapisując każdy ukończony krok eksperymentu."""

    if epochs <= 0:
        raise ValueError("Liczba epok musi być dodatnia.")
    if config != load_run_config(run_path):
        raise ValueError("Konfiguracja została zmieniona; utwórz nowy eksperyment.")

    loaders = (train_loader, val_loader)

    if resume:
        state = restore_training(
            run_path, model, optimizer, scheduler, config, loaders, device
        )
    else:
        if any(
            (run_path / name).exists() for name in ("last.pt", "best.pt", "history.csv")
        ):
            raise FileExistsError(
                "Katalog ma już trening; wybierz resume lub nowy run."
            )
        state = initial_training_state()

    if state["epoch"] >= epochs:
        print(
            f"Osiągnięto już {state['epoch']} epok; cel {epochs} nie wymaga treningu."
        )
        return state["history"]

    for epoch in range(state["epoch"] + 1, epochs + 1):
        started = perf_counter()
        learning_rate = optimizer.param_groups[0]["lr"]

        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, device, show_progress
        )
        val_loss = validate(model, val_loader, criterion, device, show_progress)

        scheduler.step()
        duration = perf_counter() - started
        record_epoch(state, train_loss, val_loss, learning_rate, duration)
        save_epoch(
            run_path, model, optimizer, scheduler, config, state, loaders, device
        )

        print(
            f"Epoka {epoch}/{epochs} · {config['loss_name']}: "
            f"train={train_loss:.6f}, val={val_loss:.6f}, "
            f"lr={learning_rate:.1e} | {duration:.1f} s",
            flush=True,
        )

    return state["history"]


Odtwarzamy model przez fabrykę na podstawie checkpointu nowego formatu, bez zależności od aktualnych ustawień nowego treningu.


In [ ]:
def load_model(path: Path, device: torch.device) -> tuple[nn.Module, dict[str, object]]:
    """Wczytaj wagi do zapisanej architektury bez potrzeby ładowania datasetu."""

    checkpoint = read_checkpoint(path)
    architecture = checkpoint["architecture"]
    model = build_model(architecture["name"], architecture["params"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()

    return model, checkpoint


def build_training_components(
    config: dict[str, object], device: torch.device
) -> tuple[
    nn.Module, nn.Module, torch.optim.Optimizer, torch.optim.lr_scheduler.StepLR
]:
    """Przygotuj model, MSE, Adama i StepLR według jednego configu."""

    architecture = config["architecture"]
    model = build_model(architecture["name"], architecture["params"]).to(device)
    criterion = nn.MSELoss(reduction="mean")
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=config["lr_step_size"], gamma=config["lr_gamma"]
    )

    return model, criterion, optimizer, scheduler


Definiujemy wspólny wariant bicubic w Pillow, aby metryki i podgląd korzystały z identycznego sposobu powiększania.

In [ ]:
def upscale_bicubic(lr: Tensor, target_size: tuple[int, int]) -> Tensor:
    """Powiększ partię CPU do rozmiaru (szerokość, wysokość), zachowując RGB."""

    return torch.stack(
        [
            TF.to_tensor(
                TF.to_pil_image(image).resize(
                    target_size,
                    Image.Resampling.BICUBIC,
                )
            )
            for image in lr
        ]
    )


Liczymy MSE, PSNR i SSIM osobno dla każdego pełnego obrazu RGB bez wycinania brzegów, z zakresem 1 oraz wynikiem ograniczonym do [0, 1] zgodnie z dokumentacją scikit-image: https://scikit-image.org/docs/stable/api/skimage.metrics.html.

In [ ]:
def image_metrics(prediction: Tensor, target: Tensor) -> dict[str, float]:
    """Oceń jeden obraz CHW w RGB; identyczny obraz ma PSNR równy nieskończoności."""

    if prediction.shape != target.shape or prediction.ndim != 3:
        raise ValueError("Oczekiwano dwóch obrazów CHW o identycznych wymiarach.")

    if not torch.isfinite(prediction).all() or not torch.isfinite(target).all():
        raise ValueError("Obrazy zawierają NaN lub nieskończoność.")

    predicted = (
        prediction.detach()
        .cpu()
        .clamp(0, 1)
        .permute(1, 2, 0)
        .numpy()
        .astype(np.float64)
    )
    expected = target.detach().cpu().permute(1, 2, 0).numpy().astype(np.float64)

    mse = float(np.mean((predicted - expected) ** 2))
    psnr = (
        float(peak_signal_noise_ratio(expected, predicted, data_range=1.0))
        if mse
        else float("inf")
    )

    ssim = structural_similarity(
        expected,
        predicted,
        data_range=1.0,
        channel_axis=-1,
        gaussian_weights=True,
        sigma=1.5,
        use_sample_covariance=False,
    )

    return {"MSE": mse, "PSNR": psnr, "SSIM": float(ssim)}


Dla każdego obrazu zapisujemy metryki obu metod i zyski względem bicubic, gdzie dodatnia różnica zawsze oznacza poprawę modelu.

In [ ]:
def compare_image(
    prediction: Tensor,
    baseline: Tensor,
    target: Tensor,
    index: int,
) -> dict[str, int | float]:
    """Oceń jedną parę i obsłuż przypadek idealnej rekonstrukcji obu metod."""

    row = {"index": index}

    for name, image in (("model", prediction), ("bicubic", baseline)):
        scores = image_metrics(image, target)
        row.update({f"{name}_{key.lower()}": value for key, value in scores.items()})

    row["mse_gain"] = row["bicubic_mse"] - row["model_mse"]
    row["ssim_gain"] = row["model_ssim"] - row["bicubic_ssim"]
    row["psnr_gain"] = (
        0.0
        if row["model_psnr"] == row["bicubic_psnr"]
        else row["model_psnr"] - row["bicubic_psnr"]
    )

    return row


Zwracamy tabelę wyników poszczególnych obrazów z pełnej oceny bez losowania kolejności, aby indeks odpowiadał pozycji w zbiorze.

In [ ]:
@torch.inference_mode()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    show_progress: bool = True,
) -> pd.DataFrame:
    """Porównaj model i bicubic, zachowując wyniki każdego obrazu osobno."""

    if not isinstance(loader.sampler, SequentialSampler) or loader.drop_last:
        raise ValueError("Ocena wymaga shuffle=False oraz drop_last=False.")

    model.eval()
    rows = []

    for lr, hr in tqdm(
        loader,
        desc="Ocena modelu i bicubic",
        unit="partia",
        leave=False,
        mininterval=0.5,
        disable=not show_progress,
    ):
        sr = model(lr.to(device, non_blocking=True)).cpu()
        bicubic = upscale_bicubic(lr, (hr.shape[-1], hr.shape[-2]))

        for prediction, baseline, target in zip(sr, bicubic, hr):
            rows.append(compare_image(prediction, baseline, target, len(rows)))

    if not rows:
        raise ValueError("Brak obrazów do oceny.")

    return pd.DataFrame(rows)


Średnie metryk obliczamy z wyników pojedynczych obrazów, dzięki czemu PSNR nadal jest średnią indywidualnych wartości w dB.

In [ ]:
def summarize_results(per_image: pd.DataFrame) -> dict[str, dict[str, float]]:
    """Zbuduj zestawienie modelu i bicubic z tabeli wyników poszczególnych dzieł."""

    return {
        label: {
            metric.upper(): float(per_image[f"{prefix}_{metric}"].mean())
            for metric in ("mse", "psnr", "ssim")
        }
        for label, prefix in (("Model", "model"), ("Bicubic", "bicubic"))
    }


Powiększamy fragment z dodatkowym kontekstem i odcinamy jego obrzeże, aby łączenia nie wprowadzały sztucznych granic.

In [ ]:
@torch.inference_mode()
def predict_tile(
    model: nn.Module,
    image: Tensor,
    bounds: tuple[int, int, int, int],
    overlap: int,
    device: torch.device,
) -> Tensor:
    """Zwróć sam środek fragmentu po predykcji z kontekstem sąsiadów."""

    top, left, bottom, right = bounds
    padded_top = max(0, top - overlap)
    padded_left = max(0, left - overlap)
    padded_bottom = min(image.shape[-2], bottom + overlap)
    padded_right = min(image.shape[-1], right + overlap)

    patch = image[:, padded_top:padded_bottom, padded_left:padded_right]
    prediction = model(patch.unsqueeze(0).to(device, non_blocking=True))[0].cpu()
    scale = model.scale_factor
    offset_y = (top - padded_top) * scale
    offset_x = (left - padded_left) * scale

    return prediction[
        :,
        offset_y : offset_y + (bottom - top) * scale,
        offset_x : offset_x + (right - left) * scale,
    ]


Dobieramy kontekst LR do wczytanej architektury i odrzucamy zbyt mały overlap, aby łączenia fragmentów nie zmieniały predykcji.


In [ ]:
@torch.inference_mode()
def predict_tiles(
    model: nn.Module,
    image: Tensor,
    device: torch.device,
    tile_size: int = 256,
    overlap: int | None = None,
    show_progress: bool = True,
) -> Tensor:
    """Ogranicz pamięć GPU, składając wynik na CPU bez mieszania sąsiednich pikseli."""

    overlap = model.required_overlap if overlap is None else overlap

    if tile_size <= 0 or overlap < model.required_overlap:
        raise ValueError(
            f"Rozmiar fragmentu musi być dodatni, a kontekst co najmniej "
            f"{model.required_overlap}."
        )

    model.eval()
    height, width = image.shape[-2:]
    scale = model.scale_factor
    result = torch.empty((3, height * scale, width * scale), dtype=image.dtype)
    positions = [
        (top, left)
        for top in range(0, height, tile_size)
        for left in range(0, width, tile_size)
    ]

    for top, left in tqdm(
        positions,
        desc="Powiększanie fragmentami",
        unit="fragment",
        leave=False,
        mininterval=0.5,
        disable=not show_progress,
    ):
        bottom = min(top + tile_size, height)
        right = min(left + tile_size, width)
        prediction = predict_tile(
            model, image, (top, left, bottom, right), overlap, device
        )
        result[:, top * scale : bottom * scale, left * scale : right * scale] = (
            prediction
        )

    return result


Powiększamy zwykły obraz w całości albo fragmentami, zachowując proporcje i ograniczając wartości dopiero po złożeniu pełnego wyniku.

In [ ]:
@torch.inference_mode()
def predict_image(
    model: nn.Module,
    image: Image.Image,
    device: torch.device,
    tile_size: int | None = None,
    overlap: int | None = None,
    show_progress: bool = True,
) -> Image.Image:
    """Powiększ RGB, opcjonalnie zmniejszając zapotrzebowanie na pamięć GPU."""

    model.eval()
    lr = TF.to_tensor(image.convert("RGB"))

    if tile_size is None:
        sr = model(lr.unsqueeze(0).to(device, non_blocking=True))[0].cpu()
    else:
        sr = predict_tiles(model, lr, device, tile_size, overlap, show_progress)

    return TF.to_pil_image(sr.clamp(0, 1))


Dobieramy kilka etykiet osi epok bez pokazywania numerów spoza rzeczywistego treningu.

In [ ]:
def epoch_ticks(first: int, last: int) -> list[int]:
    """Ogranicz etykiety osi do istniejących epok i uwzględnij oba końce."""

    locator = MaxNLocator(nbins=5, integer=True)
    ticks = [
        int(value)
        for value in locator.tick_values(first, max(first + 1, last))
        if first <= value <= last
    ]

    if first not in ticks:
        ticks.insert(0, first)
    if last not in ticks:
        ticks.append(last)

    return ticks


Rozdzielamy rysowanie panelu strat od całego wykresu i ograniczamy liczbę etykiet epok, żeby zachować czytelność dłuższego treningu.

In [ ]:
def plot_loss_panel(
    axis: Axes,
    history: dict[str, list[float]],
    start_epoch: int,
    title: str,
) -> None:
    """Pokaż wybrany zakres epok z etykietami bezpośrednio przy liniach."""

    epochs = np.arange(start_epoch, len(history["train_loss"]) + 1)
    marker = "o" if len(epochs) == 1 else None

    train_offset = -12 if history["train_loss"][-1] <= history["val_loss"][-1] else 12

    for key, label, color, offset in (
        ("train_loss", "Trening", GRAY, train_offset),
        ("val_loss", "Walidacja", BLUE, -train_offset),
    ):
        values = history[key][start_epoch - 1 :]
        axis.plot(epochs, values, color=color, linewidth=2, marker=marker)
        axis.annotate(
            label,
            (epochs[-1], values[-1]),
            xytext=(8, offset),
            textcoords="offset points",
            va="center",
            color=color,
        )

    best = int(np.argmin(history["val_loss"])) + 1
    if best >= start_epoch:
        axis.scatter(best, history["val_loss"][best - 1], color=BLUE, s=35, zorder=3)

    margin = max((epochs[-1] - epochs[0]) * 0.2, 1.0)
    axis.set(
        title=title,
        xlabel="Epoka",
        ylabel="Strata · MSE",
        xlim=(epochs[0] - 0.3, epochs[-1] + margin),
    )
    axis.set_xticks(epoch_ticks(int(epochs[0]), int(epochs[-1])))
    axis.yaxis.set_major_locator(MaxNLocator(nbins=5, min_n_ticks=3))
    style_axis(axis)


Pokazujemy cały trening i osobne zbliżenie od piątej epoki, jasno oznaczając różne zakresy oraz najlepszy wynik walidacji.

In [ ]:
def plot_history(
    history: dict[str, list[float]],
    detail_start: int = 5,
) -> None:
    """Ujawnij późniejsze zmiany strat bez ukrywania początku treningu."""

    count = len(history["train_loss"])
    detail_start = max(1, min(detail_start, count))
    best = int(np.argmin(history["val_loss"])) + 1
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

    plot_loss_panel(axes[0], history, 1, "Cały trening")
    plot_loss_panel(
        axes[1],
        history,
        detail_start,
        f"Zbliżenie · od epoki {detail_start}",
    )
    fig.suptitle(
        f"Najlepsza walidacja: {history['val_loss'][best - 1]:.5f} · epoka {best}",
        x=0.05,
        ha="left",
        color=TEXT,
        fontsize=13,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.90))
    plt.show()


W osobnym niewielkim wykresie pokazujemy learning rate faktycznie użyty podczas każdej epoki.

In [ ]:
def plot_learning_rate(history: dict[str, list[float]]) -> None:
    """Pokaż harmonogram kroku uczenia, jeśli zapisano go w historii."""

    if "learning_rate" not in history:
        return

    epochs = np.arange(1, len(history["learning_rate"]) + 1)
    fig, axis = plt.subplots(figsize=(9, 2.6))
    axis.step(epochs, history["learning_rate"], where="post", color=BLUE, linewidth=2)
    axis.set(
        title="Learning rate użyty w kolejnych epokach",
        xlabel="Epoka",
        ylabel="Learning rate",
    )
    axis.set_xticks(epoch_ticks(1, len(epochs)))
    axis.yaxis.set_major_locator(MaxNLocator(nbins=4))
    style_axis(axis)
    fig.tight_layout()
    plt.show()


Definiujemy prosty panel porównujący dwie wartości metryki z bezpośrednimi etykietami i jawną obsługą nieskończonego PSNR.

In [ ]:
def plot_metric_panel(
    axis: Axes,
    results: dict[str, dict[str, float]],
    metric: str,
    title: str,
) -> None:
    """Pokaż model i bicubic na wspólnej skali jednej metryki."""

    values = [results[method][metric] for method in ("Model", "Bicubic")]
    finite = [value for value in values if np.isfinite(value)]
    low = min(0.0, min(finite, default=0.0))
    high = max(finite, default=1.0)
    span = max(high - low, 0.01)
    axis.set_xlim(low - 0.08 * span, high + 0.45 * span)

    for row, (value, color) in enumerate(zip(values, (BLUE, GRAY))):
        if np.isfinite(value):
            axis.scatter(value, row, color=color, s=65, zorder=3)
            text = f"{value:.5f}" if metric == "MSE" else f"{value:.3f}"
            axis.annotate(
                text,
                (value, row),
                xytext=(8, 0),
                textcoords="offset points",
                va="center",
                color=color,
            )
        else:
            axis.text(
                0.95,
                row,
                "∞",
                transform=axis.get_yaxis_transform(),
                ha="right",
                va="center",
                color=color,
            )

    axis.set(
        title=title, ylim=(1.6, -0.6), yticks=[0, 1], yticklabels=["Model", "Bicubic"]
    )
    axis.xaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=2))
    style_axis(axis, grid_axis="x")


Zestawiamy MSE, PSNR i SSIM w trzech małych panelach, utrzymując znaczenie kolorów i podając kierunek lepszego wyniku.

In [ ]:
def plot_metrics(
    results: dict[str, dict[str, float]],
    title: str = "Porównanie na pełnej walidacji",
) -> None:
    """Porównaj metryki o różnych jednostkach na osobnych osiach."""

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.1), sharey=True)
    panels = (
        ("MSE", "MSE · mniej = lepiej"),
        ("PSNR", "PSNR [dB] · więcej = lepiej"),
        ("SSIM", "SSIM · więcej = lepiej"),
    )

    for axis, (metric, label) in zip(axes, panels):
        plot_metric_panel(axis, results, metric, label)

    fig.suptitle(title, x=0.05, ha="left", color=TEXT, fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.88))
    plt.show()


Pokazujemy rozkład zmian PSNR względem bicubic oraz odsetek obrazów z poprawą, zamiast opierać ocenę wyłącznie na średniej.

In [ ]:
def plot_gains(per_image: pd.DataFrame) -> None:
    """Pokaż, jak często i o ile model poprawia PSNR na pojedynczych obrazach."""

    gains = per_image["psnr_gain"].to_numpy()
    finite = gains[np.isfinite(gains)]
    win_rate = 100 * float((gains > 0).mean())
    fig, axis = plt.subplots(figsize=(9, 3.8))

    if finite.size:
        extent = max(float(np.abs(finite).max()), 0.05)
        counts, edges = np.histogram(finite, bins=np.linspace(-extent, extent, 31))
        axis.bar(
            edges[:-1],
            counts,
            width=np.diff(edges),
            align="edge",
            color=[GRAY if edge < 0 else BLUE for edge in edges[:-1]],
        )

    axis.axvline(0, color=TEXT, linewidth=1)
    axis.set(
        title=f"Model poprawia PSNR dla {win_rate:.1f}% obrazów",
        xlabel="Zmiana PSNR względem bicubic [dB] · dodatnia = poprawa",
        ylabel="Liczba obrazów",
    )
    if finite.size != gains.size:
        axis.text(
            0.98,
            0.95,
            f"Poza histogramem: {gains.size - finite.size} wartości ±∞",
            transform=axis.transAxes,
            ha="right",
            va="top",
            color=GRAY,
        )

    style_axis(axis)
    fig.tight_layout()
    plt.show()


Przygotowujemy obrazy do inspekcji wspólną funkcją, aby pełny podgląd i zbliżenia używały tego samego sposobu predykcji i bicubic.

In [ ]:
@torch.inference_mode()
def comparison_images(
    model: nn.Module,
    dataset: Dataset[tuple[Tensor, Tensor]],
    index: int,
    device: torch.device,
) -> tuple[Tensor, Tensor, Tensor, Tensor]:
    """Zwróć LR, bicubic, SR i HR na CPU dla tego samego dzieła."""

    model.eval()
    lr, hr = dataset[index]
    sr = model(lr.unsqueeze(0).to(device, non_blocking=True)).cpu()[0].clamp(0, 1)
    bicubic = upscale_bicubic(lr.unsqueeze(0), (hr.shape[-1], hr.shape[-2]))[0]

    return lr, bicubic, sr, hr


Wybieramy do zbliżeń fragment z dużą zmianą jasności w HR, aby oglądać kontury i fakturę zamiast zawsze środka obrazu.

In [ ]:
def find_detail(image: Tensor, size: int) -> tuple[int, int]:
    """Znajdź wyrazisty fragment na prostej siatce bez korzystania z wyniku modelu."""

    height, width = image.shape[-2:]
    if not 2 <= size <= min(height, width):
        raise ValueError("Fragment musi mieć co najmniej 2 piksele i mieścić się w HR.")

    gray = TF.rgb_to_grayscale(image)[0]
    edges = (gray[1:, :-1] - gray[:-1, :-1]).abs()
    edges += (gray[:-1, 1:] - gray[:-1, :-1]).abs()
    best_score = -1.0
    best_position = (0, 0)

    for top in range(0, height - size + 1, max(size // 2, 1)):
        for left in range(0, width - size + 1, max(size // 2, 1)):
            score = edges[top : top + size - 1, left : left + size - 1].mean().item()
            if score > best_score:
                best_score = score
                best_position = (top, left)

    return best_position


Wycinamy dokładnie ten sam obszar z bicubic, modelu i HR, wybierając środek albo fragment wskazany przez krawędzie obrazu referencyjnego.

In [ ]:
def crop_comparison(
    images: tuple[Tensor, ...], size: int, mode: str
) -> tuple[Tensor, ...]:
    """Przygotuj porównywalne zbliżenia bez niezależnego wybierania fragmentów."""

    height, width = images[-1].shape[-2:]
    if not 2 <= size <= min(height, width):
        raise ValueError("Fragment musi mieć co najmniej 2 piksele i mieścić się w HR.")

    if mode == "edges":
        top, left = find_detail(images[-1], size)
    elif mode == "center":
        top, left = (height - size) // 2, (width - size) // 2
    else:
        raise ValueError("Wybierz detail_mode='center' albo 'edges'.")

    return tuple(image[:, top : top + size, left : left + size] for image in images[1:])


Rysujemy pełne obrazy lub odpowiadające sobie wyraziste fragmenty, z możliwością opisania grupy najlepszych i najsłabszych przypadków.

In [ ]:
def show_predictions(
    model: nn.Module,
    dataset: Dataset[tuple[Tensor, Tensor]],
    indices: list[int],
    device: torch.device,
    detail_size: int | None = None,
    detail_mode: str = "edges",
    title: str | None = None,
) -> None:
    """Porównaj całe obrazy lub identyczne fragmenty bez zmiany skali kolorów."""

    if not indices:
        raise ValueError("Wybierz co najmniej jeden obraz.")

    titles = (
        ("LR", "Bicubic", "Model", "HR")
        if detail_size is None
        else ("Bicubic", "Model", "HR")
    )
    fig, axes = plt.subplots(
        len(indices), len(titles), figsize=(14, 3.2 * len(indices)), squeeze=False
    )

    for row, index in enumerate(indices):
        images = comparison_images(model, dataset, index, device)

        if detail_size is not None:
            images = crop_comparison(images, detail_size, detail_mode)

        for axis, image, column_title in zip(axes[row], images, titles):
            axis.imshow(
                image.permute(1, 2, 0).numpy(), interpolation="nearest", vmin=0, vmax=1
            )
            if row == 0:
                axis.set_title(
                    column_title, color=BLUE if column_title == "Model" else TEXT
                )
            axis.axis("off")

        axes[row, 0].text(
            -0.04,
            0.5,
            f"#{index}",
            transform=axes[row, 0].transAxes,
            ha="right",
            va="center",
            color=GRAY,
        )

    detail_label = "krawędzie" if detail_mode == "edges" else "środek"
    label = (
        "Pełne obrazy"
        if detail_size is None
        else f"Ten sam fragment {detail_size}×{detail_size} · {detail_label}"
    )
    fig.suptitle(title or label, x=0.05, ha="left", fontsize=13, color=TEXT)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()


Zapisujemy metryki każdego obrazu i średnie z numerem ocenianej epoki, aby późniejsze wznowienie nie zacierało pochodzenia wyników.

In [ ]:
def save_evaluation(
    run_path: Path,
    split: str,
    per_image: pd.DataFrame,
    checkpoint: dict[str, object],
) -> dict[str, dict[str, float]]:
    """Zachowaj wyniki i metryki MSE, PSNR i SSIM dla ocenianego modelu."""

    results = summarize_results(per_image)
    exported = per_image.assign(checkpoint_epoch=checkpoint["epoch"])
    save_csv(exported, run_path / f"{split}_per_image.csv")

    summary = pd.DataFrame.from_dict(results, orient="index").rename_axis("method")
    summary = summary.reset_index()
    summary["checkpoint_epoch"] = checkpoint["epoch"]
    summary["architecture"] = checkpoint["architecture"]["name"]
    summary["loss_name"] = checkpoint["config"]["loss_name"]
    save_csv(summary, run_path / f"{split}_summary.csv")

    return results


Dla train tworzymy ustawienia nowego eksperymentu, a dla resume i evaluate pobieramy architekturę, ustawienia treningu i dane z config.json wskazanego katalogu.

In [ ]:
if RUN_MODE not in ("train", "resume", "evaluate"):
    raise ValueError("RUN_MODE musi być równy 'train', 'resume' albo 'evaluate'.")

if RUN_MODE == "train":
    run_config = {
        "format_version": 1,
        "architecture": {
            "name": MODEL_NAME,
            "params": {
                "scale_factor": SCALE_FACTOR,
                **MODEL_PARAMS,
            },
        },
        "loss_name": "MSE",
        "loss_reduction": "mean",
        "loss_clamp_prediction": False,
        "seed": SEED,
        "hr_size": HR_SIZE,
        "batch_size": BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "train_limit": TRAIN_LIMIT,
        "val_limit": VAL_LIMIT,
        "augment_train": AUGMENT_TRAIN,
        "num_workers": NUM_WORKERS,
        "initial_target_epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "optimizer": "Adam",
        "scheduler": "StepLR",
        "lr_step_size": LR_STEP_SIZE,
        "lr_gamma": LR_GAMMA,
        "dataset_path": str(DATASET_PATH.resolve()),
        "split_method": "matg_80_10_10_v1",
        "metrics": {
            "color_space": "RGB",
            "data_range": 1.0,
            "clamp_prediction": True,
            "border_crop": 0,
            "psnr_aggregation": "mean_per_image_db",
            "ssim_gaussian_weights": True,
            "ssim_sigma": 1.5,
            "ssim_use_sample_covariance": False,
        },
    }
else:
    if RUN_PATH is None:
        raise ValueError("Ustaw RUN_PATH na katalog eksperymentu.")

    run_path = Path(RUN_PATH).expanduser()

    if not run_path.is_absolute():
        run_path = project_root / run_path

    run_path = run_path.resolve()
    run_config = load_run_config(run_path)

seed_everything(run_config["seed"], DEVICE)

if RUN_MODE == "train":
    run_config["environment"] = environment_info(DEVICE)

print(f"Tryb: {RUN_MODE}; urządzenie: {DEVICE}; loss: {run_config['loss_name']}")


Odtwarzamy wspólny podział i stałe podzbiory zgodnie z konfiguracją, sprawdzając fingerprint oraz liczebności przed użyciem istniejącego eksperymentu.


In [ ]:
ds_full = load_wikiart(Path(run_config["dataset_path"]))
ds_train, ds_val, ds_test = split_wikiart(ds_full, run_config["seed"])

train_limit = run_config["train_limit"]
val_limit = run_config["val_limit"]

if train_limit is not None and train_limit <= 0:
    raise ValueError("TRAIN_LIMIT musi być dodatni albo równy None.")
if val_limit is not None and val_limit <= 0:
    raise ValueError("VAL_LIMIT musi być dodatni albo równy None.")

train_count = len(ds_train) if train_limit is None else min(train_limit, len(ds_train))
val_count = len(ds_val) if val_limit is None else min(val_limit, len(ds_val))
ds_train_used = ds_train.select(range(train_count))
ds_val_fit = ds_val.select(range(val_count))

data_info = {
    "dataset_fingerprint": ds_full._fingerprint,
    "dataset_count": len(ds_full),
    "train_count": train_count,
    "val_count": val_count,
    "full_val_count": len(ds_val),
    "test_count": len(ds_test),
}

if RUN_MODE == "train":
    run_config.update(data_info)
elif any(run_config.get(key) != value for key, value in data_info.items()):
    raise ValueError("Dane lub ich podział różnią się od zapisanego eksperymentu.")

print(f"Wspólny podział: train={len(ds_train)}, val={len(ds_val)}, test={len(ds_test)}")
print(f"Uczenie: {train_count}; walidacja robocza: {val_count}; pełna: {len(ds_val)}")


Augmentację włączamy tylko w treningowym datasecie, a wszystkie podziały, próbkę walidacyjną i deterministyczną ocenę zachowujemy bez zmian.

In [ ]:
scale = run_config["architecture"]["params"]["scale_factor"]
hr_size = run_config["hr_size"]
seed = run_config["seed"]
workers = run_config["num_workers"]
eval_batch_size = run_config["eval_batch_size"]

if workers != 0:
    raise ValueError("W tym schemacie wznowienia używamy NUM_WORKERS=0.")

train_dataset = SuperResolutionDataset(
    ds_train_used, hr_size, scale, augment=run_config["augment_train"]
)
val_fit_dataset = SuperResolutionDataset(ds_val_fit, hr_size, scale)
val_dataset = SuperResolutionDataset(ds_val, hr_size, scale)
test_dataset = SuperResolutionDataset(ds_test, hr_size, scale)

train_loader = make_loader(train_dataset, run_config["batch_size"], True, seed, workers)
val_fit_loader = make_loader(val_fit_dataset, eval_batch_size, False, seed, workers)
val_loader = make_loader(val_dataset, eval_batch_size, False, seed, workers)
test_loader = make_loader(test_dataset, eval_batch_size, False, seed, workers)

lr_batch, hr_batch = next(iter(val_fit_loader))
assert lr_batch.shape[1:] == (3, hr_size // scale, hr_size // scale)
assert hr_batch.shape[1:] == (3, hr_size, hr_size)

for batch in (lr_batch, hr_batch):
    assert batch.dtype == torch.float32 and torch.isfinite(batch).all()
    assert 0 <= batch.min().item() <= batch.max().item() <= 1


Przygotowujemy model, MSE, Adama i StepLR, tworząc nowy katalog wyłącznie w trybie train.


In [ ]:
if RUN_MODE != "evaluate":
    model, criterion, optimizer, scheduler = build_training_components(
        run_config, DEVICE
    )

if RUN_MODE == "train":
    run_path = create_run(EXPERIMENTS_PATH, run_config)

print(f"Katalog eksperymentu: {run_path}")
print(f"Architektura: {run_config['architecture']}")


Uruchamiamy nowy trening albo kontynuujemy last.pt do łącznego celu EPOCHS, a w trybie evaluate pomijamy uczenie.


In [ ]:
if RUN_MODE in ("train", "resume"):
    history = fit(
        model,
        train_loader,
        val_fit_loader,
        criterion,
        optimizer,
        scheduler,
        DEVICE,
        EPOCHS,
        run_path,
        run_config,
        resume=RUN_MODE == "resume",
        show_progress=SHOW_PROGRESS,
    )


Do oceny wybieramy best.pt, a pełną historię odtwarzamy z last.pt i zapisujemy do history.csv także po ponownym otwarciu notebooka.


In [ ]:
last_checkpoint = read_checkpoint(run_path / "last.pt")

if last_checkpoint["config"] != run_config:
    raise ValueError("Konfiguracja katalogu nie odpowiada checkpointowi.")

recover_run_artifacts(run_path, last_checkpoint)
best_model, checkpoint = load_model(run_path / "best.pt", DEVICE)
history = last_checkpoint["history"]

print(
    f"Najlepsza epoka: {checkpoint['epoch']} / {last_checkpoint['epoch']}; "
    f"walidacyjne {checkpoint['config']['loss_name']}: {checkpoint['val_loss']:.6f}"
)


Pokazujemy pełny przebieg strat i czytelne zbliżenie późniejszych epok, z ograniczoną liczbą etykiet na osi czasu.

In [ ]:
plot_history(history)


Sprawdzamy na wykresie, w których epokach scheduler zmniejszał krok uczenia.

In [ ]:
plot_learning_rate(history)


Oceniamy najlepszy model na pełnej walidacji i zapisujemy metryki poszczególnych obrazów oraz średnie we wspólnym katalogu uruchomienia.


In [ ]:
val_per_image = evaluate(best_model, val_loader, DEVICE, show_progress=SHOW_PROGRESS)
val_per_image["filename"] = list(ds_val["filename"])
val_results = save_evaluation(run_path, "validation", val_per_image, checkpoint)

plot_metrics(val_results, title=f"Pełna walidacja · {len(val_dataset)} obrazów")


Sprawdzamy odsetek poprawionych obrazów i rozkład różnic PSNR, aby ocenić, czy poprawa średniej dotyczy większości dzieł.

In [ ]:
plot_gains(val_per_image)


Oglądamy stałe przykłady walidacyjne po treningu, aby porównywać te same dzieła w kolejnych eksperymentach.

In [ ]:
show_predictions(best_model, val_dataset, PREVIEW_INDICES, DEVICE)


Oglądamy te same wyraziste fragmenty bicubic, modelu i HR wybrane według krawędzi obrazu referencyjnego.

In [ ]:
show_predictions(best_model, val_dataset, PREVIEW_INDICES[:2], DEVICE, detail_size=64)


Wybieramy największe i najmniejsze poprawy PSNR względem bicubic oraz pokazujemy małą tabelę z nazwami obrazów.

In [ ]:
best_cases = val_per_image.nlargest(CASE_COUNT, "psnr_gain")
worst_cases = val_per_image.nsmallest(CASE_COUNT, "psnr_gain")
best_indices = best_cases["index"].astype(int).tolist()
worst_indices = worst_cases["index"].astype(int).tolist()

case_table = pd.concat(
    [
        best_cases.assign(group="Najlepsze przypadki"),
        worst_cases.assign(group="Najsłabsze przypadki"),
    ]
)
display(case_table[["group", "index", "filename", "psnr_gain"]].round(3))


Oglądamy wyraziste fragmenty obrazów, na których model uzyskał największy wzrost PSNR względem bicubic.

In [ ]:
show_predictions(
    best_model,
    val_dataset,
    best_indices,
    DEVICE,
    detail_size=64,
    title="Najlepsze przypadki · różnica PSNR względem bicubic",
)


Oglądamy obrazy z najmniejszą poprawą lub pogorszeniem PSNR, aby szukać powtarzających się słabości modelu.

In [ ]:
show_predictions(
    best_model,
    val_dataset,
    worst_indices,
    DEVICE,
    detail_size=64,
    title="Najsłabsze przypadki · różnica PSNR względem bicubic",
)


Pokazujemy użycie modelu na oryginale 256×256 jako powiększanie do 512×512 bez referencji HR, więc ten przykład służy tylko ocenie wizualnej.

In [ ]:
input_image = ds_val[PREVIEW_INDICES[0]]["image"].convert("RGB")
output_image = predict_image(
    best_model,
    input_image,
    DEVICE,
    tile_size=INFERENCE_TILE_SIZE,
    overlap=INFERENCE_OVERLAP,
    show_progress=SHOW_PROGRESS,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for axis, image, title in zip(
    axes,
    (input_image, output_image),
    (
        f"Wejście · {input_image.width}×{input_image.height}",
        f"Model · {output_image.width}×{output_image.height}",
    ),
):
    axis.imshow(image, interpolation="nearest")
    axis.set_title(title)
    axis.axis("off")

fig.tight_layout()
plt.show()


Końcowy test pozostaje wyłączony, a po jego świadomym włączeniu zapisujemy wyniki poszczególnych obrazów oraz pokazujemy średnie metryki i rozkład poprawy.

In [ ]:
if RUN_TEST:
    test_per_image = evaluate(
        best_model, test_loader, DEVICE, show_progress=SHOW_PROGRESS
    )
    test_per_image["filename"] = list(ds_test["filename"])
    test_results = save_evaluation(run_path, "test", test_per_image, checkpoint)

    plot_metrics(test_results, title=f"Końcowy test · {len(test_dataset)} obrazów")
    plot_gains(test_per_image)


Powiększamy wskazane zdjęcie najlepszym modelem i porównujemy je wizualnie z bicubic, ponieważ nie mamy referencji HR.


In [ ]:
test_image_path = project_root / "notebooks/igoh/testowa_fota_2.jpg"

with Image.open(test_image_path) as image:
    test_image = image.convert("RGB")

test_result = predict_image(
    best_model,
    test_image,
    DEVICE,
    tile_size=INFERENCE_TILE_SIZE,
    overlap=INFERENCE_OVERLAP,
    show_progress=SHOW_PROGRESS,
)
test_bicubic = test_image.resize(test_result.size, Image.Resampling.BICUBIC)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
previews = (
    (test_image, f"Wejście · {test_image.width}×{test_image.height}"),
    (test_bicubic, f"Bicubic · {test_bicubic.width}×{test_bicubic.height}"),
    (test_result, f"Model · {test_result.width}×{test_result.height}"),
)

for axis, (image, title) in zip(axes, previews):
    axis.imshow(image, interpolation="nearest")
    axis.set_title(title)
    axis.axis("off")

fig.tight_layout()
plt.show()
